In [2]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Pauline\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [101]:
pip install webdriver-manager


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Pauline\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [102]:
pip install bs4


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: C:\Users\Pauline\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import random
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
import csv
import requests
import pandas as pd
import re

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import os


In [34]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Modo headless
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


# CODIGO PARA CREAR WINE_DATA
- Cuidado, que de momento cada vez que se ejecuta, el wine_data.csv aplasta lo de antes, hay que cambiar nombre del csv al final y guardarlo si lo queremos conservar.
- A ver si para despues lo cambio para que las filas se agreguen a lo que ya esta o si es mas comodo tener un csv x tipo de vino. 
- Depende también como proceso los datos, si los junto todo antes en un solo txt o si voy poco a poco

**Falta integrar lo que se hara con Selenium:**
- Caracteristicas
- Nota de sabor 

**Test hecho con 500 blancos:** 7 min, parece que el año no se encuentra siempre pero no supe mejoralo más :( 


In [ ]:
# Leer los enlaces desde un archivo de texto
with open(r'C:\Users\Pauline\Downloads\Proyecto_grupo2_vinos\tintos\def_tintos_merged_150_to_202.txt', 'r', encoding='utf-8-sig') as f:
    urls = f.readlines()  # Lee todos los enlaces en el archivo

# Limitar a los primeros 5 enlaces
#urls = urls[:2]

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar una lista para guardar todos los datos
all_wine_data = []

# Recorrer cada URL en la lista
for url in urls:
    url = url.strip()  # Eliminar cualquier espacio en blanco o salto de línea

    try:
        # Solicitud a la página
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Inicializar datos
        wine_data = {}

        # País, región, bodega, tipo de vino, uva
        breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
        country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
        region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
        winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
        wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
        #grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'


        #Grape
        grapes = soup.find_all("a", class_="anchor_anchor__m8Qi- wineFacts__link--3aTg9")
        grape = [grape.text.strip() for grape in grapes if "grapes" in grape["href"]]
        grape = ', '.join(grape) if grape else 'No disponible'


        # Nombre del vino 
        wine_headline = soup.find('div', class_='wineHeadline-module__wineHeadline--32Ety')
        if wine_headline:
            # Tomamos todo el texto dentro del bloque, sin intentar separarlo
            name = wine_headline.get_text(strip=True)
        else:
            name = 'No disponible'
        
            # Si el nombre del vino contiene el nombre de la bodega, eliminamos la bodega del nombre
        if winery.lower() in name.lower():
            name = name.replace(winery, '').strip()


        # Año
        button_elements = soup.find_all('button', class_='MuiButtonBase-root')
        year = 'No disponible'

            # Buscar en los botones primero
        for button in button_elements:
            if button.get('aria-label') and button.get('aria-label').isdigit() and len(button.get('aria-label').strip()) == 4:
                year = button.get('aria-label').strip()
                break

            # Si no se encuentra en los botones, buscar en el span con la clase 'vivino-mui-14ngluw-componentChildren'
        if year == 'No disponible':
            year_element = soup.find('span', class_='vivino-mui-14ngluw-componentChildren')
            if year_element and year_element.text.strip().isdigit() and len(year_element.text.strip()) == 4:
                year = year_element.text.strip()

            # Si aún no se ha encontrado, buscar todos los 'span' con la clase 'vintageListRow__year--34Tuc' y tomar el primero
        if year == 'No disponible':
            vintage_section = soup.find('div', id='vintageListSection')
            if vintage_section:
                # Buscar todos los 'span' dentro del div y filtrar aquellos que contienen un año (4 dígitos)
                year_elements = vintage_section.find_all('span', string=re.compile(r'\d{4}'))
                if year_elements:
                    year = year_elements[0].string.strip()  # Usamos .string para obtener solo el texto


        # Precio
        price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
        if price_element:
            price = price_element.text.replace('€', '').replace('\xa0', '').strip()
        else:
            # Si no lo encuentra, busca el precio en la segunda clase
            price_element = soup.find(class_='purchaseAvailabilityPPC__amount--2_4GT')
            if price_element:
                # Utilizamos regex para encontrar el precio con la coma
                match = re.search(r'\d{1,3}(?:,\d{3})*(?:\.\d+)?', price_element.text)
                if match:
                    price = match.group(0).replace('\xa0', '').strip()
                else:
                    price = 'No disponible'
            else:
                price = 'No disponible'

        
        # Grados de Alcohol
        alcohol_element = soup.find(class_='wineFacts__wineFacts--2Ih8B')
            # Buscar todos los spans dentro de la tabla
        if alcohol_element:
            spans = alcohol_element.find_all('span')
            # Filtrar los spans que contienen un número seguido de '%' (grado de alcohol)
        alcohol = 'No disponible'
        for span in spans:
                # Usamos una expresión regular para buscar un número seguido de '%'
                match = re.search(r'\d+%', span.text.strip())
                if match:
                    alcohol = match.group(0)  # El valor que coincide con la expresión regular
                    alcohol = alcohol.replace('%', '')
                    break  # Detener la búsqueda cuando encontramos el primer grado de alcohol
        else:
            alcohol = 'No disponible'



        # Notas de sabor
        taste_containers = soup.find_all(class_='slider__viewPort--30MrB')
        taste_notes = []
        for container in taste_containers:
            # Encontrar todos los elementos con la clase 'tasteNote__popularKeywords--1gIa2'
            taste_keywords = container.find_all(class_='tasteNote__popularKeywords--1gIa2')

            # Recorrer cada uno de los elementos encontrados y extraer el texto
            for keyword in taste_keywords:
                if keyword.text.strip():  # Solo si no está vacío
                    taste_notes.append(keyword.text.strip())

        # Unir todas las notas en una sola cadena, separada por coma
        taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

        
        # Valoración
        rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
        rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

        # Maridajes
        food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
        pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]


        # Guardar los datos de esta URL
        wine_data = {
            'Url': url,
            'ID': re.search(r'\/(\d+)$', url).group(1),
            'Nombre': name,
            'Año': year,
            'País': country,
            'Región': region,
            'Bodega': winery,
            'Tipo de vino': wine_type,
            'Uva': grape,
            'Precio': price,
            'Valoración': rating,
            'Contenido de alcohol': alcohol,
            'Maridajes':', '.join(pairings),
            
        }
        all_wine_data.append(wine_data)

    except Exception as e:
        print(f"Error al procesar la URL {url}: {e}")

# Guardar los resultados en un archivo CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
    writer.writeheader()
    writer.writerows(all_wine_data)

# Convertir los datos a un DataFrame y mostrar los primeros registros
df = pd.DataFrame(all_wine_data)
pd.set_option('display.max_colwidth', None)  # No limita el ancho de las celdas
pd.set_option('display.max_rows', None)  # Sin límite de filas
pd.set_option('display.max_columns', None)  # Sin límite de columnas

print(df.head())


Error al procesar la URL Url: Invalid URL 'Url': No scheme supplied. Perhaps you meant https://Url?
Error al procesar la URL https://www.vivino.com/ES/es/rigas-sparkling-medium-dry/w/1157882: 'NoneType' object has no attribute 'text'
Error al procesar la URL https://www.vivino.com/ES/es/skepparps-vingard-grand-prix-solaris-mousserande-brut/w/6175956: 'NoneType' object has no attribute 'text'
Error al procesar la URL https://www.vivino.com/ES/es/divo-brut-veneto-sparkling-v-epuq3/w/3111172: 'NoneType' object has no attribute 'text'
                                                                                 Url  \
0           https://www.vivino.com/ES/es/vinos-sanz-fri-sanz-te-semi-dulce/w/4559173   
1   https://www.vivino.com/ES/es/castellblanc-castellblanch-cava-brut-cava/w/2453836   
2  https://www.vivino.com/ES/es/palacio-de-bornos-verdejo-5-5deg-frizzante/w/2329608   
3   https://www.vivino.com/ES/es/palacio-de-bornos-rosado-5-5deg-frizzante/w/4243274   
4                      

## CODIGOS QUE FALTAN INTEGRAR ##

In [40]:
# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/domaine-raymond-usseglio-and-fils-la-genese/w/9716238"

# Navegar a la página
driver.get(url)

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Diccionario para guardar los resultados
progress_values = {}

try:
    # Buscar todas las barras de progreso
    progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
    # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
    if len(progress_elements) == len(labels):
        # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
        for i, element in enumerate(progress_elements):
            style = element.get_attribute('style')
            print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
            # Sacar el valor del left :
            valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
            valor_left = round(float(valor_left)/10,1)
            progress_values[labels[i]] = valor_left
        print(progress_values)

            #si el valor de left es 0, no hacemos nada, 
            # si el valor del left no es 0 a 0, tnemos que coger el valor que hay antes del % y /10 y redondearlo para que tenga 1 solo decimal
            #el resultado de eso, hay que asignarlo a cada una de las labels
            
            
    else:
        print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
except Exception as e:
    print(f"Error durante la extracción: {e}")



Barra 1: width: 20%; left: 41.9107%;
Barra 2: width: 20%; left: 59.7054%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 52.5%;
{'Ligero/Poderoso': 4.2, 'Suave/Tánico': 6.0, 'Seco/Dulce': 0.0, 'Débil/Ácido': 5.2}


In [41]:
#CARACTERISTICAS VINOS
 
# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/domaine-raymond-usseglio-and-fils-la-genese/w/9716238"

# Navegar a la página
driver.get(url)

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Diccionario para guardar los resultados
progress_values = {}

try:
    # Buscar todas las barras de progreso
    progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
    # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
    if len(progress_elements) == len(labels):
        # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
        for i, element in enumerate(progress_elements):
            style = element.get_attribute('style')
            print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
            # Sacar el valor del left :
            valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
            valor_left = round(float(valor_left)/10,1)
            progress_values[labels[i]] = valor_left
            
    else:
        print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
except Exception as e:
    print(f"Error durante la extracción: {e}")

# Nombre del archivo CSV
csv_filename = "tintos/tintos_test.csv"
# Cargar datos existentes o crear un nuevo DataFrame
if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename)
else:
    df = pd.DataFrame(columns=["ID"] + labels)
# Convertir ID a string para evitar errores de tipo
df["ID"] = df["ID"].astype(str)
# Verificar si la ID ya existe en el CSV

ID = url.split("/w/")[1].split("?")[0] if "/w/" in url else "Desconocido"

if ID in df["ID"].values:
    df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
    print(f"Datos actualizados para el vino con ID: {ID}")
else:
    df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
    print(f"Nuevo registro agregado con ID: {ID}")
# Guardar de nuevo el archivo CSV
df.to_csv(csv_filename, index=False)
print(f"Datos guardados en {csv_filename}")

# Cerrar el navegador
driver.quit()



Barra 1: width: 20%; left: 41.9107%;
Barra 2: width: 20%; left: 59.7054%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 52.5%;
Datos actualizados para el vino con ID: 9716238
Datos guardados en tintos/tintos_test.csv


Procesando URL: https://www.vivino.com/ES/es/celler-aixala-alcait-destrankis/w/1873838
Barra 1: width: 20%; left: 79.2007%;
Barra 2: width: 20%; left: 51.5551%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 47.9231%;
Nuevo registro agregado con ID: 1873838
Procesando URL: https://www.vivino.com/ES/es/les-freses-tallaruques/w/7753698


C:\Users\Pauline\AppData\Local\Temp\ipykernel_26628\2767437383.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 44.8077%;
Barra 2: width: 20%; left: 56.8462%;
Barra 3: width: 20%; left: 18.75%;
Barra 4: width: 20%; left: 43.8846%;
Nuevo registro agregado con ID: 7753698
Procesando URL: https://www.vivino.com/ES/es/puiggros-signes-vinyes-velles/w/1201118
Barra 1: width: 20%; left: 59.5596%;
Barra 2: width: 20%; left: 42.914%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 46.029%;
Nuevo registro agregado con ID: 1201118
Procesando URL: https://www.vivino.com/ES/es/caroline-and-marcel-gisclard-aramon/w/5498856
Barra 1: width: 20%; left: 44.1937%;
Barra 2: width: 20%; left: 54.5491%;
Barra 3: width: 20%; left: 7.37037%;
Barra 4: width: 20%; left: 64.9566%;
Nuevo registro agregado con ID: 5498856
Procesando URL: https://www.vivino.com/ES/es/lungarotti-il-pometo-sangiovese/w/7251518
Barra 1: width: 20%; left: 41.3833%;
Barra 2: width: 20%; left: 36.877%;
Barra 3: width: 20%; left: 23.2358%;
Barra 4: width: 20%; left: 44.308%;
Nuevo registro agregado con ID: 7251

C:\Users\Pauline\AppData\Local\Temp\ipykernel_26628\2767437383.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 54.0453%;
Barra 2: width: 20%; left: 39.8185%;
Barra 3: width: 20%; left: 10.9015%;
Barra 4: width: 20%; left: 58.6291%;
Nuevo registro agregado con ID: 4481721
Procesando URL: https://www.vivino.com/ES/es/emiliana-natura-pinot-noir/w/2683748
Barra 1: width: 20%; left: 36.2523%;
Barra 2: width: 20%; left: 1.17464%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 46.706%;
Nuevo registro agregado con ID: 2683748
Procesando URL: https://www.vivino.com/ES/es/casa-vinicola-gioacchino-garofoli-spa-kerria-lacrima-di-morro-d-alba/w/1321322
Barra 1: width: 20%; left: 36.6021%;
Barra 2: width: 20%; left: 33.0048%;
Barra 3: width: 20%; left: 9.99654%;
Barra 4: width: 20%; left: 47.0592%;
Nuevo registro agregado con ID: 1321322
Procesando URL: https://www.vivino.com/ES/es/cantina-tollo-mo-montepulciano-d-abruzzo-riserva/w/3287552
Barra 1: width: 20%; left: 40.7313%;
Barra 2: width: 20%; left: 38.038%;
Barra 3: width: 20%; left: 4.97504%;
Barra 4: width: 20%; l

...

Barra 1: width: 20%; left: 56.4692%;
Barra 2: width: 20%; left: 25.4708%;
Barra 3: width: 20%; left: 18.2467%;
Barra 4: width: 20%; left: 33.9426%;
Nuevo registro agregado con ID: 83183
Procesando URL: https://www.vivino.com/ES/es/finca-moncloa-vinedos-propios/w/1186088
Barra 1: width: 20%; left: 60.5424%;
Barra 2: width: 20%; left: 41.192%;
Barra 3: width: 20%; left: 12.246%;
Barra 4: width: 20%; left: 43.063%;
Nuevo registro agregado con ID: 1186088
Procesando URL: https://www.vivino.com/ES/es/fattoria-di-magliano-sinarra/w/1446756
Barra 1: width: 20%; left: 37.3043%;
Barra 2: width: 20%; left: 40.6112%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 46.9279%;
Nuevo registro agregado con ID: 1446756
Procesando URL: https://www.vivino.com/ES/es/d-ventura-vina-caneiro/w/1424908
Barra 1: width: 20%; left: 46.1498%;
Barra 2: width: 20%; left: 38.4961%;
Barra 3: width: 20%; left: 8.01503%;
Barra 4: width: 20%; left: 62.7561%;
Nuevo registro agregado con ID: 1424908
Procesando 

C:\Users\Pauline\AppData\Local\Temp\ipykernel_26628\2767437383.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 55.4818%;
Barra 2: width: 20%; left: 71.3908%;
Barra 3: width: 20%; left: 19.0995%;
Barra 4: width: 20%; left: 51.8847%;
Nuevo registro agregado con ID: 2045032
Procesando URL: https://www.vivino.com/ES/es/luca-ferraris-del-martin-barbera-d-asti/w/21412
Barra 1: width: 20%; left: 51.568%;
Barra 2: width: 20%; left: 25.4742%;
Barra 3: width: 20%; left: 2.21106%;
Barra 4: width: 20%; left: 64.9902%;
Nuevo registro agregado con ID: 21412
Procesando URL: https://www.vivino.com/ES/es/tramuntana-siurell/w/6675879
Barra 1: width: 20%; left: 57.1013%;
Barra 2: width: 20%; left: 32.6971%;
Barra 3: width: 20%; left: 12.5776%;
Barra 4: width: 20%; left: 44.906%;
Nuevo registro agregado con ID: 6675879
Procesando URL: https://www.vivino.com/ES/es/martin-berdugo-crianza/w/1150140
Barra 1: width: 20%; left: 59.0902%;
Barra 2: width: 20%; left: 52.1939%;
Barra 3: width: 20%; left: 12.4157%;
Barra 4: width: 20%; left: 55.4997%;
Nuevo registro agregado con ID: 1150140
Procesa

C:\Users\Pauline\AppData\Local\Temp\ipykernel_26628\2767437383.py:76: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 57.4617%;
Barra 2: width: 20%; left: 48.4661%;
Barra 3: width: 20%; left: 13.7257%;
Barra 4: width: 20%; left: 39.8278%;
Nuevo registro agregado con ID: 4460515
Procesando URL: https://www.vivino.com/ES/es/neil-ellis-cabernet-sauvignon/w/88267
Barra 1: width: 20%; left: 71.8113%;
Barra 2: width: 20%; left: 59.5594%;
Barra 3: width: 20%; left: 8.26644%;
Barra 4: width: 20%; left: 56.7726%;
Nuevo registro agregado con ID: 88267
Procesando URL: https://www.vivino.com/ES/es/fattoria-nittardi-ad-astra-maremma-toscana/w/74858
Barra 1: width: 20%; left: 47.8236%;
Barra 2: width: 20%; left: 44.3569%;
Barra 3: width: 20%; left: 1.74444%;
Barra 4: width: 20%; left: 45.4447%;
Nuevo registro agregado con ID: 74858
Procesando URL: https://www.vivino.com/ES/es/chateau-de-saint-cosme-cotes-du-rhone-les-deux-albion/w/1173569
Barra 1: width: 20%; left: 64.6036%;
Barra 2: width: 20%; left: 48.8896%;
Barra 3: width: 20%; left: 4.97066%;
Barra 4: width: 20%; left: 52.0306%;
Nuev

Completo y correcto para vinos tintos:

In [6]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time
import re
import pandas as pd

# Function to save URLs to a file
def save_urls_to_file(urls, batch_number):
    filename = f'caracteristicas_tintos{batch_number}.csv'
    with open(filename, 'w') as file:
        for url in urls:
            file.write(url + '\n')
    print(f"Las nuevas URLs han sido guardadas en '{filename}'.")

# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Headless mode
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('C:\\Users\\Pauline\\Downloads\\Proyecto_grupo2_vinos\\tintos\\urls_limpios.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]


# List to store new URLs
batch_size = 100
batch_number = 1
start_index = 0



# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]


# Process each URL batch in the list
while start_index < len(urls_list):
    # Slice the URLs for the current batch
    url_batch = urls_list[start_index:start_index + batch_size]
    start_index += batch_size

    # Prepare the DataFrame for this batch
    df = pd.DataFrame(columns=["ID"] + labels)



 # Process each URL in the batch
for original_url in url_batch:
    print(f"Procesando URL: {original_url}")  # Mostrar progreso
    match = re.search(r'\/(\d+)$', original_url)  # Buscar el ID en la URL
    ID = match.group(1) if match else "Desconocido"  # Si no se encuentra, asignar "Desconocido"

    try:
        # Open the URL
        driver.get(original_url)

        # Wait for the page to load completely
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))

        # Buscar todas las barras de progreso
        progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
        # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
        if len(progress_elements) == len(labels):
            progress_values = {}  # Diccionario para los valores de un vino
            
            # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
            for i, element in enumerate(progress_elements):
                style = element.get_attribute('style')
                print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                # Sacar el valor del left:
                valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                valor_left = round(float(valor_left)/10, 1)  # Ajustar el valor a 1 decimal
                progress_values[labels[i]] = valor_left

            # Convertir ID a string para evitar errores de tipo
            progress_values["ID"] = ID
            
            # Añadir el registro al DataFrame
            df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
            print(f"Nuevo registro agregado con ID: {ID}")

        else:
            print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
        
    except Exception as e:
        print(f"Error durante la extracción: {e}")

# Guardar los resultados de este lote en un archivo CSV
batch_filename = f"caracteristicas_tintos_{batch_number}.csv"
df.to_csv(batch_filename, index=False)
print(f"Datos guardados en {batch_filename}")

# Incrementar el número de lote para el siguiente
batch_number += 1

# Cerrar el navegador
driver.quit()


Procesando URL: https://www.vivino.com/ES/es/micro-bio-microbio-ruf-y-ann/w/7775295
Barra 1: width: 20%; left: 49.4829%;
Barra 2: width: 20%; left: 39.4737%;
Barra 3: width: 20%; left: 0px;
Barra 4: width: 20%; left: 45.0512%;
Nuevo registro agregado con ID: 7775295
Procesando URL: https://www.vivino.com/ES/es/revoltier-and-fils-chateauneuf-du-pape/w/1218559


C:\Users\Pauline\AppData\Local\Temp\ipykernel_26628\2024988775.py:90: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)


Barra 1: width: 20%; left: 80%;
Barra 2: width: 20%; left: 38.6857%;
Barra 3: width: 20%; left: 6.6854%;
Barra 4: width: 20%; left: 38.6775%;
Nuevo registro agregado con ID: 1218559
Procesando URL: https://www.vivino.com/ES/es/marco-real-comunidad-foral-de-navarra-finca-la-pared-graciano/w/11107985
Barra 1: width: 20%; left: 66.407%;
Barra 2: width: 20%; left: 39.1027%;
Barra 3: width: 20%; left: 10.2524%;
Barra 4: width: 20%; left: 38.029%;
Nuevo registro agregado con ID: 11107985
Procesando URL: https://www.vivino.com/ES/es/kornell-marith-blauburgunder/w/1876690
Barra 1: width: 20%; left: 28.3566%;
Barra 2: width: 20%; left: 22.2649%;
Barra 3: width: 20%; left: 5.53797%;
Barra 4: width: 20%; left: 52.0824%;
Nuevo registro agregado con ID: 1876690
Procesando URL: https://www.vivino.com/ES/es/la-cave-winemakers-selection-cape-blend/w/1794584
Barra 1: width: 20%; left: 80%;
Barra 2: width: 20%; left: 55.7433%;
Barra 3: width: 20%; left: 12.0565%;
Barra 4: width: 20%; left: 41.3305%;
Nue

Codigo ok para tintos, con batch de 100 :

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time
import re
import pandas as pd

# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Headless mode
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('C:\\Users\\Pauline\\Downloads\\Proyecto_grupo2_vinos\\tintos\\ids_no_en_segundo_csv.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Variables for handling batch processing
batch_size = 167  # Number of URLs per batch
batch_number = 1
start_index = 0

# Process each URL batch in the list
while start_index < len(urls_list):
    # Slice the URLs for the current batch
    url_batch = urls_list[start_index:start_index + batch_size]
    start_index += batch_size

    # Prepare the DataFrame for this batch
    df = pd.DataFrame(columns=["ID"] + labels)

    # Process each URL in the batch
    for original_url in url_batch:
        print(f"Procesando URL: {original_url}")  # Mostrar progreso
        match = re.search(r'\/(\d+)$', original_url)  # Buscar el ID en la URL
        ID = match.group(1) if match else "Desconocido"  # Si no se encuentra, asignar "Desconocido"

        try:
            # Open the URL
            driver.get(original_url)

            # Wait for the page to load completely
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))

            # Buscar todas las barras de progreso
            progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
        
            # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
            if len(progress_elements) == len(labels):
                progress_values = {}  # Diccionario para los valores de un vino
                
                # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
                for i, element in enumerate(progress_elements):
                    style = element.get_attribute('style')
                    print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                    # Sacar el valor del left:
                    valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                    valor_left = round(float(valor_left)/10, 1)  # Ajustar el valor a 1 decimal
                    progress_values[labels[i]] = valor_left

                # Convertir ID a string para evitar errores de tipo
                progress_values["ID"] = ID
                
                # Añadir el registro al DataFrame
                df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
                print(f"Nuevo registro agregado con ID: {ID}")

            else:
                print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
            
        except Exception as e:
            print(f"Error durante la extracción: {e}")

    # Guardar los resultados de este lote en un archivo CSV
    batch_filename = f"caracteristicas_tintos_{batch_number}.csv"
    df.to_csv(batch_filename, index=False)
    print(f"Datos guardados en {batch_filename}")

    # Incrementar el número de lote para el siguiente
    batch_number += 1

# Cerrar el navegador
driver.quit()


Procesando URL: URL: https://www.vivino.com/ES/es/fleur-de-lisse-fontfleurie-saint-emilion-grand-cru/w/9926210
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/casta-de-vinos-casta-tinta-domina/w/7783134
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/cl-de-martino-viejas-tinajas-cinsault/w/1241009
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/makarounas-boutique-winery-maratheftiko/w/6917981
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/fautor-cabernet-sauvignon-merlot/w/5032035
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/roberto-henriquez-rivera-del-notro-tinto/w/3

ESPUMOSOS:

In [1]:
#Barras de sabor con batch

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time
import re
import pandas as pd

# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Headless mode
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('C:\\Users\\Pauline\\Downloads\\Proyecto_grupo2_vinos\\espumosos\\all_espumosos.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]

# Limitar a las primeras 5 URLs para prueba (puedes quitar esta línea cuando quieras procesar todas)
# urls_list = urls_list[:5]

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Débil/Ácido", "Amable/Con Burbujas"]

# Nombre del archivo CSV
csv_filename = "caracteristicas_espumosos.csv"

# Cargar datos existentes o crear un nuevo DataFrame
if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename)
    # Extraer los IDs ya procesados
    processed_ids = df["ID"].values
    # Filtrar las URLs para comenzar después de las que ya fueron procesadas
    urls_list = [url for url in urls_list if re.search(r'\/(\d+)$', url).group(1) not in processed_ids]
else:
    df = pd.DataFrame(columns=["ID"] + labels)
    
# Tamaño del batch
batch_size = 100  # Puedes ajustar el tamaño del lote según necesites
num_batches = (len(urls_list) // batch_size) + (1 if len(urls_list) % batch_size > 0 else 0)

# Procesar en batches
for batch_index in range(num_batches):
    batch_urls = urls_list[batch_index * batch_size : (batch_index + 1) * batch_size]
    print(f"\n🚀 Procesando batch {batch_index + 1} de {num_batches}...")

    for original_url in batch_urls:
        print(f"Procesando URL: {original_url}")  # Mostrar progreso
        match = re.search(r'\/(\d+)$', original_url)  # Buscar el ID en la URL
        ID = match.group(1) if match else "Desconocido"  # Si no se encuentra, asignar "Desconocido"

        try:
            # Open the URL
            driver.get(original_url)

            # Wait for the page to load completely
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))

            # Buscar todas las barras de progreso
            progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
        
            # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
            if len(progress_elements) == len(labels):
                progress_values = {}  # Diccionario para los valores de un vino
                
                # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
                for i, element in enumerate(progress_elements):
                    style = element.get_attribute('style')
                    print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                    # Sacar el valor del left:
                    valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                    valor_left = round(float(valor_left)/10, 1)  # Ajustar el valor a 1 decimal
                    progress_values[labels[i]] = valor_left

                # Convertir ID a string para evitar errores de tipo
                progress_values["ID"] = ID

                # Verificar si ya existe el ID en el DataFrame
                if ID in df["ID"].values:
                    df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
                    print(f"Datos actualizados para el vino con ID: {ID}")
                else:
                    df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
                    print(f"Nuevo registro agregado con ID: {ID}")

            else:
                print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
        
        except Exception as e:
            print(f"Error durante la extracción: {e}")

    # Guardar el DataFrame después de procesar cada batch
    df.to_csv(csv_filename, index=False)
    print(f"Datos del batch {batch_index + 1} guardados en {csv_filename}")

    # Pausa entre batches para evitar bloqueos
    time.sleep(5)

# Cerrar el navegador
driver.quit()

print("\n🎯 ¡Todos los batches han sido procesados y los datos guardados correctamente! 🚀")


🚀 Procesando batch 1 de 55...
Procesando URL: ï»¿https://www.vivino.com/ES/es/fr-simonnet-febvre-cremant-de-bourgogne-cuvee-s-brut/w/3967508
Error durante la extracción: Message: invalid argument
  (Session info: chrome=133.0.6943.142)
Stacktrace:
	GetHandleVerifier [0x00007FF641D0C6A5+28789]
	(No symbol) [0x00007FF641C75B20]
	(No symbol) [0x00007FF641B08DCC]
	(No symbol) [0x00007FF641AF5B30]
	(No symbol) [0x00007FF641AF3F2E]
	(No symbol) [0x00007FF641AF459C]
	(No symbol) [0x00007FF641B0CDEA]
	(No symbol) [0x00007FF641BB069E]
	(No symbol) [0x00007FF641B8732A]
	(No symbol) [0x00007FF641BAF7E3]
	(No symbol) [0x00007FF641B87103]
	(No symbol) [0x00007FF641B4FFC0]
	(No symbol) [0x00007FF641B51273]
	GetHandleVerifier [0x00007FF642051AED+3458237]
	GetHandleVerifier [0x00007FF64206829C+3550316]
	GetHandleVerifier [0x00007FF64205DB9D+3507565]
	GetHandleVerifier [0x00007FF641DD2C6A+841274]
	(No symbol) [0x00007FF641C809EF]
	(No symbol) [0x00007FF641C7CB34]
	(No symbol) [0x00007FF641C7CCD6]
	(No

KeyboardInterrupt: 

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import os
import time



# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Headless mode
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize the WebDriver
driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('C:\\Users\\Pauline\\Downloads\\Proyecto_grupo2_vinos\\tintos\\ids_no_en_segundo_csv.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Débil/Ácido", "Amable/Con Burbujas"]

# Diccionario para guardar los resultados
progress_values = {}

# Process each URL in the list
for original_url in urls_list:
    print(f"Procesando URL: {original_url}")  # Show progress

    try:
        # Open the URL
        driver.get(original_url)

        # Wait for the page to load completely (adjust the condition as needed)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
        #time.sleep(3)

        # Buscar todas las barras de progreso
        progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
        # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
        if len(progress_elements) == len(labels):
            # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
            for i, element in enumerate(progress_elements):
                style = element.get_attribute('style')
                print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                # Sacar el valor del left :
                valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                valor_left = round(float(valor_left)/10,1)
                progress_values[labels[i]] = valor_left
            
        else:
            print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
    except Exception as e:
        print(f"Error durante la extracción: {e}")

# Nombre del archivo CSV
csv_filename = "def_tintos.csv"
# Cargar datos existentes o crear un nuevo DataFrame
if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename)
else:
    df = pd.DataFrame(columns=["ID"] + labels)
# Convertir ID a string para evitar errores de tipo
df["ID"] = df["ID"].astype(str)
# Verificar si la ID ya existe en el CSV

ID = url.split("/w/")[1].split("?")[0] if "/w/" in url else "Desconocido"

if ID in df["ID"].values:
    df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
    print(f"Datos actualizados para el vino con ID: {ID}")
else:
    df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
    print(f"Nuevo registro agregado con ID: {ID}")
# Guardar de nuevo el archivo CSV
df.to_csv(csv_filename, index=False)
print(f"Datos guardados en {csv_filename}")

# Cerrar el navegador
driver.quit()

Procesando URL: URL: https://www.vivino.com/ES/es/fleur-de-lisse-fontfleurie-saint-emilion-grand-cru/w/9926210
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/casta-de-vinos-casta-tinta-domina/w/7783134
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/cl-de-martino-viejas-tinajas-cinsault/w/1241009
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/makarounas-boutique-winery-maratheftiko/w/6917981
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/fautor-cabernet-sauvignon-merlot/w/5032035
El número de barras de progreso no coincide con el número de etiquetas esperadas.
Procesando URL: URL: https://www.vivino.com/ES/es/roberto-henriquez-rivera-del-notro-tinto/w/3

Exception ignored in: <function Service.__del__ at 0x000001E21158FEC0>
Traceback (most recent call last):
  File "C:\Users\Pauline\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\selenium\webdriver\common\service.py", line 200, in __del__
    self.stop()
  File "C:\Users\Pauline\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\selenium\webdriver\common\service.py", line 157, in stop
    self.send_remote_shutdown_command()
  File "C:\Users\Pauline\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\selenium\webdriver\common\service.py", line 137, in send_remote_shutdown_command
    request.urlopen(f"{self.service_url}/shutdown")
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\urllib\request.py", line 216, in

NameError: name 'url' is not defined

In [10]:
#CARACTERISTICAS ESPUMOSOS:


# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/bollinger-vieilles-vignes-francaises-blanc-de-noirs-brut-champagne/w/18938?year=2009&price_id=30860996"

# Navegar a la página
driver.get(url)

labels = ["Ligero/Poderoso", "Débil/Ácido", "Amable/Con Burbujas"]

# Diccionario para guardar los resultados
progress_values = {}

try:
    # Buscar todas las barras de progreso
    progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
    # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
    if len(progress_elements) == len(labels):
        # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
        for i, element in enumerate(progress_elements):
            left_value = element.get_attribute('style').split('left: ')[1].split('%')[0] if 'left' in element.get_attribute('style') else None
            
            if left_value:
                # Convertir a float y mapearlo con los valores predefinidos
                value = round(float(left_value) / 10, 1)
                # Asignar la etiqueta correspondiente con los valores predefinidos
                progress_values[labels[i]] = value
                print(f"{labels[i]}: {value}")
            else:
                print(f"No se encontró el atributo 'left' para {labels[i]}.")
    else:
        print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
except Exception as e:
    print(f"Error durante la extracción: {e}")

# Cerrar el navegador
driver.quit()


Ligero/Poderoso: 7.0
Débil/Ácido: 7.3
Amable/Con Burbujas: 7.0


# URL uno por uno
Todo ok salvo nota de sabor


In [71]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd

# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/mestres-coquet-gran-reserva-brut-nature/w/4311421"

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Solicitud a la página
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

# Inicializar datos
wine_data = {}

try:
    # Nombre del vino: Ahora extraemos solo el texto del nombre (después de la bodega)
    wine_headline = soup.find('div', class_='wineHeadline-module__wineHeadline--32Ety')
    if wine_headline:
        # Obtener todo el texto dentro del div, luego buscar solo la parte que es el nombre del vino
        text_parts = wine_headline.get_text(strip=True).split(" ")  # Separar por espacios
        # Todo lo que venga después del primer elemento (que sería la bodega)
        name = " ".join(text_parts[1:])  # Tomamos todo después del primer elemento (la bodega)
    else:
        name = 'No disponible'

    #Año :
    # Buscar todos los botones en la página
    button_elements = soup.find_all('button', class_='MuiButtonBase-root')

    # Inicializar año como 'No disponible'
    year = 'No disponible'

    # Iterar sobre todos los botones y buscar uno que contenga un año (4 dígitos)
    for button in button_elements:
        # Verificamos si el aria-label o el texto del botón contiene un año válido
        if button.get('aria-label') and button.get('aria-label').isdigit() and len(button.get('aria-label').strip()) == 4:
            year = button.get('aria-label').strip()
            break  # Si encontramos el año, terminamos el bucle

    if year == 'No disponible':
    # Si no encontramos el año en los botones, intentamos otra estrategia (como se hizo anteriormente)
        year_element = soup.find('span', class_='vivino-mui-14ngluw-componentChildren')
        if year_element and year_element.text.strip().isdigit() and len(year_element.text.strip()) == 4:
            year = year_element.text.strip()




    # País, región, bodega, tipo de vino, uva
    breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
    country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
    region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
    winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
    wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
    grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'

    # Precio
    price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
    price = price_element.text.replace('€', '').replace('\xa0', '').strip() if price_element else 'No disponible'

    # Valoración
    rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
    rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

    # Notas de sabor
    taste_containers = soup.find_all(class_='tasteNote__textContainer--2xPXc')
    taste_notes = []
    for container in taste_containers:
        taste_keywords = container.find(class_='tasteNote__popularKeywords--1gIa2')
        if taste_keywords:
            taste_notes.append(taste_keywords.text.strip())
    taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

    # Maridajes
    food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
    pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]



    # Guardar datos
    wine_data = {
        'Nombre': name,
        'Año': year,
        'País': country,
        'Región': region,
        'Bodega': winery,
        'Tipo de vino': wine_type,
        'Uva': grape,
        'Precio': price,
        'Valoración': rating,
        'Notas de sabor': taste_notes,
        'Maridajes': ', '.join(pairings),
         }

except Exception as e:
    print(f"Error durante la extracción: {e}")

# Guardar en CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=wine_data.keys())
    writer.writeheader()
    writer.writerow(wine_data)

# Convertir a DataFrame y mostrar los primeros registros
pd.set_option('display.max_colwidth', None)  # No limita el ancho de las celdas
pd.set_option('display.max_rows', None)   # Sin límite de filas
pd.set_option('display.max_columns', None)  # Sin límite de columnas

df = pd.DataFrame([wine_data])
print(df.head())



                     Nombre   Año    País Región   Bodega   Tipo de vino  \
0  Gran Reserva Brut Nature  2019  España   Cava  Mestres  Vino espumoso   

      Uva Precio Valoración Notas de sabor  \
0  Mezcla  15.87        3.9  No disponible   

                                                                    Maridajes  
0  Marisco, Aperitivos y tentempiés, Pescado blanco, Aperitivo, Carne adobada  


# OTRO CODIGO DE VICENTE
Texto definitivo para coger las url de archivo txt

In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import pandas as pd
import time

# Headers para evitar bloqueos
headers = {
    'User -Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar lista para almacenar todos los datos de vino
all_wine_data = []

# Leer las URLs desde un archivo de texto
with open('def_espumoso200.txt', 'r') as file:
    urls = file.readlines()

# Iterar sobre cada URL
for index, url in enumerate(urls, start=1):
    url = url.strip()  # Eliminar espacios en blanco
    wine_data = {}  # Inicializar datos para cada vino

    print(f"Procesando URL {index}/{len(urls)}: {url}")  # Mostrar el progreso

    try:
        # Solicitud a la página
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Nombre y año
        wine_headline = soup.find(class_='wineHeadline-module__vintage--1UHSo')
        if wine_headline:
            name = wine_headline.find('a').text.strip() if wine_headline.find('a') else 'No disponible'
            year = wine_headline.text.strip().split()[-1]  # Última palabra debería ser el año
        else:
            name = 'No disponible'
            year = 'No disponible'

        # País, región, bodega, tipo de vino, uva
        breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
        country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
        region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
        winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
        wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
        grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'

        # Precio
        price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
        price = price_element.text.replace('€', '').replace('\xa0', '').strip() if price_element else 'No disponible'

        # Valoración
        rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
        rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

        # Notas de sabor
        taste_containers = soup.find_all(class_='tasteNote__textContainer--2xPXc')
        taste_notes = []
        for container in taste_containers:
            taste_keywords = container.find(class_='tasteNote__popularKeywords--1gIa2')
            if taste_keywords:
                taste_notes.append(taste_keywords.text.strip())
        taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

        # Maridajes
        food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
        pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]

        # Guardar datos
        wine_data = {
            'Nombre': name,
            'Año': year,
            'País': country,
            'Región': region,
            'Bodega': winery,
            'Tipo de vino': wine_type,
            'Uva': grape,
            'Precio': price,
            'Valoración': rating,
            'Notas de sabor': taste_notes,
            'Maridajes': ', '.join(pairings),
        }

        all_wine_data.append(wine_data)  # Agregar datos a la lista

    except Exception as e:
        print(f"Error durante la extracción de {url}: {e}")

    # Esperar 2 segundos antes de la siguiente solicitud
    time.sleep(2)

# Guardar todos los datos en un CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
    writer.writeheader()
    writer.writerows(all_wine_data)

# Convertir a DataFrame y mostrar los primeros registros
df = pd.DataFrame(all_wine_data)
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'def_espumoso200.txt'